# CSE 438 · Part B — SimCLR → DeepLabV3 (LiTS liver+tumour, 3-class)

**Group 01 · Dept. of CSE, East West University**

| # | Name | Student ID |
|---|------|-----------|
| 1 | Md. Asif Hossain | 2022-3-60-007 |
| 2 | Nabil Subhan | 2022-3-60-063 |
| 3 | K M Nudar | 2022-3-60-234 |

**Pipeline:** ImageNet-init **ResNet-50** → **SimCLR** contrastive pretraining (50 ep, images only, the Part-A
*train* split) → transfer the encoder into **DeepLabV3-ResNet50** (our Part-A best decoder) → supervised
fine-tuning (50 ep) on the small labelled **val** split → evaluate on the **test** split. 3 classes:
`bg=0, liver=1, tumor=2`. Attach: **LiTS 256×256** + **Part-B data-prep output** (`partB_split_roles.json`).

In [ ]:
# ===== Setup, CUDA probe, AMP, config =====
import os, re, json, time, math, random, copy, warnings
from pathlib import Path
from contextlib import nullcontext
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50
from tqdm.auto import tqdm
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CONFIG = dict(
    ssl_epochs=50, seg_epochs=50, ssl_batch=96, seg_batch=8,
    ssl_size=224, seg_size=256, num_classes=3,
    ssl_lr=3e-4, seg_lr=1e-4, weight_decay=1e-2, temperature=0.2,
    proj_hidden=512, proj_dim=128, ce_weights=[0.3, 1.0, 6.0],
    tumor_oversample=4.0, method="simclr",
)
CLASS_NAMES = ["background", "liver", "tumor"]
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)

def select_device():
    if not torch.cuda.is_available():
        print("CUDA unavailable -> CPU, AMP off."); return torch.device("cpu"), False
    try:
        x = torch.randn(2,3,32,32,device="cuda"); l = nn.Conv2d(3,8,3,padding=1).to("cuda")
        l(x).float().mean().backward(); torch.cuda.synchronize()
        print("CUDA probe passed:", torch.cuda.get_device_name(0)); return torch.device("cuda"), True
    except Exception as e:
        print("CUDA visible but kernel probe failed:", type(e).__name__, "-> CPU."); return torch.device("cpu"), False

device, AMP = select_device()
SCALER = torch.amp.GradScaler("cuda", enabled=AMP)
def amp_ctx(): return torch.autocast("cuda", dtype=torch.float16) if AMP else nullcontext()
print("device", device, "| AMP", AMP)
print("CONFIG", json.dumps(CONFIG))

## 1. Load the reused Part-A split (SSL roles) + LiTS loaders

In [ ]:
# ===== Load partB_split_roles.json (train=unlabelled, val=labelled, test=eval) + raw LiTS =====
roles_paths = list(Path("/kaggle/input").rglob("partB_split_roles.json"))
assert roles_paths, "Attach the Part-B data-prep output (partB_split_roles.json)."
roles_meta = json.load(open(roles_paths[0]))
roles = roles_meta["roles"]
print("roles:", {k: len(v) for k, v in roles.items()})

imgdir = Path(roles_meta["dirs"]["images"])
if imgdir.exists():
    IMG_DIR, LIVER_DIR, TUMOR_DIR = imgdir, Path(roles_meta["dirs"]["liver"]), Path(roles_meta["dirs"]["tumor"])
else:
    root = next(iter(Path("/kaggle/input").rglob("Thesis_data")), Path("/kaggle/input"))
    IMG_DIR   = next(p for p in root.iterdir() if p.is_dir() and "image" in p.name.lower())
    LIVER_DIR = next(p for p in root.iterdir() if p.is_dir() and "liver" in p.name.lower())
    TUMOR_DIR = next(p for p in root.iterdir() if p.is_dir() and "tumor" in p.name.lower())

def vskey(p):
    n = [int(x) for x in re.findall(r"\d+", Path(p).stem)]; return (n[0], n[1]) if len(n) >= 2 else (n[0], -1)
img_by = {vskey(p): p for p in IMG_DIR.glob("*.png")}
liv_by = {vskey(p): p for p in LIVER_DIR.glob("*.png")}
tum_by = {vskey(p): p for p in TUMOR_DIR.glob("*.png")}

def load_25d(vol, sl):
    c = np.array(Image.open(img_by[(vol, sl)]).convert("L"))
    p = img_by.get((vol, sl-1)); n = img_by.get((vol, sl+1))
    pp = np.array(Image.open(p).convert("L")) if p is not None else c
    nn_ = np.array(Image.open(n).convert("L")) if n is not None else c
    return np.stack([pp, c, nn_], axis=-1)
def fuse_label(vol, sl):
    liv = np.array(Image.open(liv_by[(vol, sl)]).convert("L")) > 0
    tum = np.array(Image.open(tum_by[(vol, sl)]).convert("L")) > 0
    lab = np.zeros(liv.shape, np.uint8); lab[liv] = 1; lab[tum] = 2; return lab
def parse_ids(ids): return [tuple(int(x) for x in s.split("_")) for s in ids]
print("images paired:", len(img_by))

## 2. Datasets — SSL two-view (images only) · labelled · eval

In [ ]:
# ===== transforms + datasets =====
ssl_view = A.Compose([
    A.RandomResizedCrop(size=(CONFIG["ssl_size"], CONFIG["ssl_size"]), scale=(0.2, 1.0)),
    A.HorizontalFlip(0.5), A.ColorJitter(0.2, 0.2, 0.2, 0.05, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
label_tf = A.Compose([
    A.HorizontalFlip(0.5), A.VerticalFlip(0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=0.06, rotate=(-15, 15), p=0.5),
    A.OneOf([A.ElasticTransform(alpha=40, sigma=6), A.GridDistortion(num_steps=5, distort_limit=0.2)], p=0.25),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.3), A.GaussNoise(p=0.2),
    A.Resize(CONFIG["seg_size"], CONFIG["seg_size"]),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
eval_tf = A.Compose([A.Resize(CONFIG["seg_size"], CONFIG["seg_size"]),
                     A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])

class SSLViewDataset(Dataset):
    def __init__(self, ids): self.ids = parse_ids(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = load_25d(*self.ids[i]); return ssl_view(image=img)["image"], ssl_view(image=img)["image"]
class LabelledDataset(Dataset):
    def __init__(self, ids, tf): self.ids = parse_ids(ids); self.tf = tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        v, s = self.ids[i]; o = self.tf(image=load_25d(v, s), mask=fuse_label(v, s)); return o["image"], o["mask"].long()

pin = device.type == "cuda"
ssl_loader = DataLoader(SSLViewDataset(roles["unlabelled_pretrain"]), batch_size=CONFIG["ssl_batch"],
                        shuffle=True, num_workers=2, pin_memory=pin, drop_last=True)
finetune_loader = DataLoader(LabelledDataset(roles["labelled_finetune"], label_tf), batch_size=CONFIG["seg_batch"],
                             shuffle=True, num_workers=2, pin_memory=pin, drop_last=False)
test_loader = DataLoader(LabelledDataset(roles["eval_monitor_test"], eval_tf), batch_size=CONFIG["seg_batch"],
                         shuffle=False, num_workers=2, pin_memory=pin, drop_last=False)
print("batches -> ssl", len(ssl_loader), "| finetune", len(finetune_loader), "| test", len(test_loader))

## 3. SimCLR model + NT-Xent (float32-stable under AMP)

In [ ]:
# ===== SimCLR ResNet-50 (ImageNet init) + projection head =====
class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            self.encoder = resnet50(weights=ResNet50_Weights.DEFAULT); print("ResNet-50 ImageNet init.")
        except Exception as e:
            print("ImageNet weights unavailable:", e); self.encoder = resnet50(weights=None)
        d = self.encoder.fc.in_features; self.encoder.fc = nn.Identity()
        self.projector = nn.Sequential(nn.Linear(d, CONFIG["proj_hidden"]), nn.ReLU(True),
                                       nn.Linear(CONFIG["proj_hidden"], CONFIG["proj_dim"]))
    def forward(self, x): f = self.encoder(x); return f, self.projector(f)

def nt_xent(z1, z2, temp):
    with torch.autocast(device_type=z1.device.type, enabled=False):
        z1 = F.normalize(z1.float(), dim=1); z2 = F.normalize(z2.float(), dim=1)
        z = torch.cat([z1, z2], 0); N = z.size(0)
        sim = (z @ z.T) / temp
        sim.masked_fill_(torch.eye(N, dtype=torch.bool, device=z.device), torch.finfo(sim.dtype).min)
        idx = torch.arange(N, device=z.device); pos = (idx + z1.size(0)) % N
        return F.cross_entropy(sim, pos)

simclr = SimCLRModel().to(device)
ssl_opt = torch.optim.AdamW(simclr.parameters(), lr=CONFIG["ssl_lr"], weight_decay=CONFIG["weight_decay"])
ssl_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ssl_opt, T_max=CONFIG["ssl_epochs"])
xb, _ = next(iter(ssl_loader))  # smoke test
with torch.no_grad(), amp_ctx():
    _, p1 = simclr(xb[:4].to(device)); _, p2 = simclr(xb[:4].to(device))
print("NT-Xent smoke loss:", float(nt_xent(p1, p2, CONFIG["temperature"])))

## 4. Stage A — SimCLR pretraining (50 epochs)

In [ ]:
# ===== SimCLR 50-epoch pretraining (per-epoch time + pretext loss) =====
ssl_hist = []
for ep in range(1, CONFIG["ssl_epochs"] + 1):
    simclr.train(); losses = []; t0 = time.time()
    for v1, v2 in tqdm(ssl_loader, desc=f"SimCLR {ep}/{CONFIG['ssl_epochs']}", leave=False):
        if v1.size(0) < 2: continue
        v1 = v1.to(device, non_blocking=pin); v2 = v2.to(device, non_blocking=pin)
        ssl_opt.zero_grad(set_to_none=True)
        with amp_ctx(): _, p1 = simclr(v1); _, p2 = simclr(v2)
        loss = nt_xent(p1, p2, CONFIG["temperature"])
        SCALER.scale(loss).backward(); SCALER.step(ssl_opt); SCALER.update()
        losses.append(loss.item())
    ssl_sched.step()
    ssl_hist.append({"epoch": ep, "loss": float(np.mean(losses)), "sec": round(time.time()-t0, 1)})
    print(f"  epoch {ep}: NT-Xent {ssl_hist[-1]['loss']:.4f} | {ssl_hist[-1]['sec']}s/epoch")
ssl_df = pd.DataFrame(ssl_hist); ssl_df.to_csv(WORK/"simclr_pretrain_history.csv", index=False)
torch.save(simclr.encoder.state_dict(), WORK/"simclr_resnet50_encoder.pt")
print("mean sec/epoch:", round(ssl_df.sec.mean(), 1), "| saved encoder")

plt.figure(figsize=(6,4)); plt.plot(ssl_df.epoch, ssl_df.loss, marker="o")
plt.title("SimCLR pretext (NT-Xent) loss"); plt.xlabel("epoch"); plt.ylabel("NT-Xent"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(WORK/"simclr_pretrain_curve.png", dpi=140); plt.show()

## 5. Stage B — transfer to DeepLabV3 + 3-class metric harness

In [ ]:
# ===== DeepLabV3 (Part-A decoder) + transfer SimCLR encoder + loss/metrics =====
seg = deeplabv3_resnet50(weights=None, weights_backbone=None, num_classes=CONFIG["num_classes"], aux_loss=True)
rep = seg.backbone.load_state_dict(simclr.encoder.state_dict(), strict=False)
print("transferred SimCLR encoder -> DeepLabV3 backbone | missing:", len(rep.missing_keys))
seg = seg.to(device)

ce_w = torch.tensor(CONFIG["ce_weights"], device=device)
def dice_loss(logits, tgt, eps=1.0):
    p = torch.softmax(logits, 1); oh = F.one_hot(tgt, CONFIG["num_classes"]).permute(0,3,1,2).float()
    inter = (p*oh).sum((0,2,3)); den = p.sum((0,2,3)) + oh.sum((0,2,3))
    return (1 - (2*inter+eps)/(den+eps)).mean()
def criterion(out, tgt):
    lg = out["out"].float(); loss = F.cross_entropy(lg, tgt, weight=ce_w) + dice_loss(lg, tgt)
    if out.get("aux") is not None: loss = loss + 0.4*F.cross_entropy(out["aux"].float(), tgt, weight=ce_w)
    return loss

class ConfMat:
    def __init__(self, n): self.n = n; self.mat = torch.zeros(n, n, dtype=torch.long)
    def update(self, pred, tgt):
        k = (tgt >= 0) & (tgt < self.n)
        self.mat += torch.bincount(self.n*tgt[k].long() + pred[k].long(),
                                   minlength=self.n**2).reshape(self.n, self.n).cpu()
    def metrics(self):
        m = self.mat.double(); tp = m.diag()
        iou = tp / (m.sum(1) + m.sum(0) - tp).clamp(min=1e-9)
        dice = 2*tp / (m.sum(1) + m.sum(0)).clamp(min=1e-9)
        tumor_sens = (m[2,2] / m[2,:].sum().clamp(min=1e-9)).item()
        return dict(mIoU=iou.mean().item(), per_class_iou=iou.tolist(), mean_dice=dice.mean().item(),
                    pixel_acc=(tp.sum()/m.sum().clamp(min=1e-9)).item(), tumor_sensitivity=tumor_sens)

@torch.inference_mode()
def evaluate(model, loader):
    model.eval(); cm = ConfMat(CONFIG["num_classes"])
    for x, y in loader:
        x = x.to(device, non_blocking=pin)
        with amp_ctx(): out = model(x)["out"]
        cm.update(out.argmax(1).cpu(), y)
    return cm.metrics(), cm
print("DeepLabV3 ready.")

## 6. Stage B — fine-tune on the labelled *val* split (50 ep), monitor on *test*

In [ ]:
# ===== 50-epoch fine-tune; checkpoint by best TEST mIoU (test doubles as val, per brief) =====
seg_opt = torch.optim.AdamW(seg.parameters(), lr=CONFIG["seg_lr"], weight_decay=CONFIG["weight_decay"])
warm = torch.optim.lr_scheduler.LinearLR(seg_opt, start_factor=0.1, total_iters=3)
cos = torch.optim.lr_scheduler.CosineAnnealingLR(seg_opt, T_max=CONFIG["seg_epochs"]-3)
seg_sched = torch.optim.lr_scheduler.SequentialLR(seg_opt, [warm, cos], milestones=[3])

seg_hist, best_miou, best_state = [], -1.0, None
for ep in range(1, CONFIG["seg_epochs"] + 1):
    seg.train(); tl = []; t0 = time.time()
    for x, y in tqdm(finetune_loader, desc=f"finetune {ep}/{CONFIG['seg_epochs']}", leave=False):
        x = x.to(device, non_blocking=pin); y = y.to(device, non_blocking=pin).long()
        seg_opt.zero_grad(set_to_none=True)
        with amp_ctx(): out = seg(x)
        loss = criterion(out, y)
        SCALER.scale(loss).backward(); SCALER.step(seg_opt); SCALER.update()
        tl.append(loss.item())
    seg_sched.step()
    tm, _ = evaluate(seg, test_loader)
    seg_hist.append({"epoch": ep, "train_loss": float(np.mean(tl)), "test_mIoU": tm["mIoU"],
                     "test_tumor_iou": tm["per_class_iou"][2], "test_tumor_sens": tm["tumor_sensitivity"],
                     "sec": round(time.time()-t0, 1)})
    print(f"  ep {ep}: loss {seg_hist[-1]['train_loss']:.4f} | test mIoU {tm['mIoU']:.4f} "
          f"| tumor IoU {tm['per_class_iou'][2]:.4f} | {seg_hist[-1]['sec']}s")
    if tm["mIoU"] > best_miou:
        best_miou = tm["mIoU"]; best_state = copy.deepcopy({k: v.cpu() for k, v in seg.state_dict().items()})
seg.load_state_dict(best_state); seg.to(device)
seg_df = pd.DataFrame(seg_hist); seg_df.to_csv(WORK/"simclr_finetune_history.csv", index=False)
print("best test mIoU:", round(best_miou, 4), "| mean sec/epoch:", round(seg_df.sec.mean(), 1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(seg_df.epoch, seg_df.train_loss, marker="o"); ax[0].set_title("fine-tune loss (val split)")
ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)
ax[1].plot(seg_df.epoch, seg_df.test_mIoU, marker="o", label="test mIoU")
ax[1].plot(seg_df.epoch, seg_df.test_tumor_iou, marker="o", label="test tumor IoU")
ax[1].set_title("monitor on test split"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(WORK/"simclr_finetune_curves.png", dpi=140); plt.show()

## 7. Test evaluation (Task E metrics) + confusion matrix

In [ ]:
# ===== final test metrics + confusion matrix + append results.json =====
final, cm = evaluate(seg, test_loader)
print("epoch-50 mIoU:", round(seg_df.test_mIoU.iloc[-1], 4), "| best-ckpt mIoU:", round(final["mIoU"], 4))
print(json.dumps({k: (round(v,4) if isinstance(v,float) else [round(x,4) for x in v])
                  for k,v in final.items()}, indent=2))

m = cm.mat.double(); mn = (m / m.sum(1, keepdim=True).clamp(min=1e-9)).numpy()
plt.figure(figsize=(5,4)); plt.imshow(mn, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3): plt.text(j, i, f"{mn[i,j]:.2f}", ha="center",
                                color="white" if mn[i,j] > .5 else "black")
plt.xticks(range(3), CLASS_NAMES, rotation=20); plt.yticks(range(3), CLASS_NAMES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("SimCLR->DeepLabV3 confusion (row-norm)")
plt.colorbar(); plt.tight_layout(); plt.savefig(WORK/"simclr_confusion.png", dpi=140); plt.show()

res_path = WORK/"results.json"
results = json.load(open(res_path)) if res_path.exists() else {}
results["simclr"] = {
    "method": "SimCLR", "encoder": "ResNet-50 (ImageNet-init, 50ep SSL)", "decoder": "DeepLabV3 ASPP",
    "n_labelled_finetune": len(roles["labelled_finetune"]), "n_pretrain": len(roles["unlabelled_pretrain"]),
    "epoch50_test_mIoU": float(seg_df.test_mIoU.iloc[-1]), "best_ckpt": final,
    "ssl_sec_per_epoch": float(ssl_df.sec.mean()), "seg_sec_per_epoch": float(seg_df.sec.mean()),
}
json.dump(results, open(res_path, "w"), indent=2)
print("appended results.json ->", list(results.keys()))

## 8. Qualitative predictions (Task F preview)

In [ ]:
# ===== qualitative grid: image | GT | prediction | overlay =====
cmap = plt.matplotlib.colors.ListedColormap(["black", "gold", "red"])
def denorm(t): return np.clip(t.permute(1,2,0).numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN), 0, 1)
tumor_ids = [s for s in roles["eval_monitor_test"]][:200]
ds = LabelledDataset(tumor_ids, eval_tf)
picks = []
for i in range(len(ds)):
    _, y = ds[i]
    if (y == 2).sum() > 40: picks.append(i)
    if len(picks) == 4: break
picks = picks or list(range(4))
fig, ax = plt.subplots(len(picks), 4, figsize=(13, 3.2*len(picks)))
seg.eval()
for r, idx in enumerate(picks):
    x, y = ds[idx]
    with torch.inference_mode(), amp_ctx(): pr = seg(x.unsqueeze(0).to(device))["out"].argmax(1)[0].cpu().numpy()
    im = denorm(x)[..., 1]
    ax[r,0].imshow(im, cmap="gray"); ax[r,0].set_title("image", fontsize=8)
    ax[r,1].imshow(y.numpy(), cmap=cmap, vmin=0, vmax=2); ax[r,1].set_title("ground truth", fontsize=8)
    ax[r,2].imshow(pr, cmap=cmap, vmin=0, vmax=2); ax[r,2].set_title("SimCLR->DeepLabV3", fontsize=8)
    ax[r,3].imshow(im, cmap="gray"); ax[r,3].imshow(pr, cmap=cmap, vmin=0, vmax=2, alpha=.45)
    ax[r,3].set_title("overlay", fontsize=8)
    for c in range(4): ax[r,c].axis("off")
plt.suptitle("SimCLR-pretrained DeepLabV3 — test predictions (liver=gold, tumor=red)", y=1.01)
plt.tight_layout(); plt.savefig(WORK/"simclr_qualitative.png", dpi=140); plt.show()

## Summary
SimCLR-pretrained ResNet-50 transferred into DeepLabV3, fine-tuned on only the **val** split (small labelled set)
and evaluated on **test**. Numbers append to `results.json`; the final-comparison notebook reads them alongside the
Part-A supervised baseline (DeepLabV3 mIoU 0.768) for the **label-efficiency** table. **BYOL / MAE / DINOv2 reuse
this exact structure** — only the pretraining stage (cells 3–4) changes.